# Partie 2 — Distributions de probabilité  
### Projet EDA — Détection de fraude bancaire

---

## 🎯 Objectifs

- Comprendre les principales **lois de probabilité** et leurs paramètres
- Identifier quelle loi modélise le mieux les variables du dataset fraude
- Appliquer des **transformations** pour normaliser des distributions asymétriques

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, expon, lognorm, bernoulli, binom

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

df = pd.read_csv("creditcard.csv")
print(f"Dataset chargé : {df.shape}")


---
## 2.1 — Loi normale (Gaussienne)

La **loi normale** $\mathcal{N}(\mu, \sigma^2)$ est la plus importante en statistiques.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**Dans notre contexte :** Les composantes `V1`–`V28` (issues d'une PCA) sont approximativement normales.


In [ ]:
# Vérification : V1, V4, V14 sont-elles normales ?
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, col in zip(axes.flatten(), ["V1", "V4", "V10", "V14", "V17", "V28"]):
    serie = df[col]
    mu, sigma = serie.mean(), serie.std()
    x = np.linspace(serie.min(), serie.max(), 200)
    
    if sns:
        sns.histplot(serie, ax=ax, stat="density", bins=50, alpha=0.5, color="#3B82F6")
    ax.plot(x, norm.pdf(x, mu, sigma), color="#EF4444", linewidth=2, label=f"N({mu:.2f}, {sigma:.2f})")
    ax.set_title(f"{col} — μ={mu:.2f}, σ={sigma:.2f}")
    ax.legend(fontsize=8)

plt.suptitle("Composantes PCA vs loi normale théorique", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("📌 Les composantes V1-V28 suivent bien une loi normale — confirmant le preprocessing PCA.")


---
## 2.2 — Loi exponentielle

La **loi exponentielle** modélise les temps entre événements.  
Elle s'adapte bien aux **délais entre transactions** (variable `Time`).

$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$


In [ ]:
# Intervalles entre transactions
inter_times = df["Time"].diff().dropna()
inter_times = inter_times[inter_times > 0]

lambda_hat = 1 / inter_times.mean()
print(f"λ estimé : {lambda_hat:.4f} transactions/seconde")
print(f"Temps moyen entre transactions : {1/lambda_hat:.2f} s")

x = np.linspace(0, inter_times.quantile(0.99), 200)

plt.figure(figsize=(9, 4))
if sns:
    sns.histplot(inter_times.clip(upper=inter_times.quantile(0.99)),
                 stat="density", bins=80, alpha=0.5, color="#3B82F6", label="Données")
plt.plot(x, expon.pdf(x, scale=1/lambda_hat), color="#EF4444", linewidth=2, label=f"Exp(λ={lambda_hat:.4f})")
plt.title("Distribution des intervalles entre transactions")
plt.xlabel("Délai (secondes)")
plt.legend()
plt.tight_layout()
plt.show()


---
## 2.3 — Loi log-normale

La **loi log-normale** s'adapte aux données positives avec une longue queue à droite.  
**`Amount` est un bon candidat** car les montants sont positifs et asymétriques.

Si $X \sim \text{LogNormal}(\mu, \sigma^2)$ alors $\ln(X) \sim \mathcal{N}(\mu, \sigma^2)$


In [ ]:
# Ajustement log-normal sur Amount (transactions non-fraude)
amount_normal = df[df["Class"] == 0]["Amount"].values
amount_normal = amount_normal[amount_normal > 0]

# Paramètres MLE
shape, loc, scale = lognorm.fit(amount_normal, floc=0)
mu_ln = np.log(scale)
sigma_ln = shape

print(f"Paramètres log-normale (MLE) :")
print(f"  μ (log scale) = {mu_ln:.4f}")
print(f"  σ (log scale) = {sigma_ln:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Original scale
x = np.linspace(0.01, np.percentile(amount_normal, 99), 300)
if sns:
    sns.histplot(amount_normal[amount_normal < np.percentile(amount_normal, 99)],
                 stat="density", bins=80, ax=axes[0], alpha=0.5, color="#3B82F6", label="Amount")
axes[0].plot(x, lognorm.pdf(x, shape, loc, scale), color="#EF4444", linewidth=2, label="Log-normale")
axes[0].set_title("Amount — ajustement log-normal")
axes[0].legend()

# Log scale
log_amount = np.log1p(amount_normal)
if sns:
    sns.histplot(log_amount, stat="density", bins=60, ax=axes[1], alpha=0.5,
                 color="#10B981", label="log(1+Amount)")
mu2, std2 = log_amount.mean(), log_amount.std()
x2 = np.linspace(log_amount.min(), log_amount.max(), 200)
axes[1].plot(x2, norm.pdf(x2, mu2, std2), color="#EF4444", linewidth=2, label="N(μ,σ) théorique")
axes[1].set_title("log(1+Amount) — approximation normale")
axes[1].legend()

plt.tight_layout()
plt.show()
print("📌 La transformation log(1+Amount) est recommandée avant modélisation ML.")


---
## 2.4 — Loi de Bernoulli

La variable `Class` suit une **loi de Bernoulli** : deux issues possibles (fraude ou non).

$$P(X=1) = p \quad \text{(probabilité de fraude)}$$


In [ ]:
# Loi de Bernoulli pour Class
p_fraud = df["Class"].mean()
print(f"p(fraude) = {p_fraud:.5f}  ({p_fraud*100:.3f}%)")
print(f"p(normal) = {1-p_fraud:.5f}  ({(1-p_fraud)*100:.3f}%)")

# Visualisation
fig, ax = plt.subplots(figsize=(6, 4))
outcomes = ["Normal (0)", "Fraude (1)"]
probs = [1 - p_fraud, p_fraud]
colors = ["#3B82F6", "#EF4444"]
bars = ax.bar(outcomes, probs, color=colors, edgecolor="white", width=0.4)
for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{prob:.4f}
({prob*100:.3f}%)", ha="center", fontsize=11)
ax.set_title(f"Distribution de Bernoulli — Class
p = {p_fraud:.5f}")
ax.set_ylabel("Probabilité")
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()


---
## 2.5 — Loi binomiale

Dans un batch de $n$ transactions, combien de fraudes attendre ?

Si chaque transaction est fraude avec probabilité $p$, le nombre de fraudes suit une **loi binomiale** $B(n, p)$.


In [ ]:
# Loi binomiale : nombre de fraudes dans un lot de n transactions
p = df["Class"].mean()
n_values = [100, 1000, 10000]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, n in zip(axes, n_values):
    k = np.arange(0, int(n * p * 5) + 1)
    pmf = binom.pmf(k, n, p)
    
    ax.bar(k, pmf, color="#7C3AED", alpha=0.8, width=0.8)
    mu_b = n * p
    sigma_b = np.sqrt(n * p * (1-p))
    ax.axvline(mu_b, color="#EF4444", linestyle="--", linewidth=1.5, label=f"E[X]={mu_b:.1f}")
    ax.set_title(f"B(n={n:,}, p={p:.4f})")
    ax.set_xlabel("Nombre de fraudes")
    ax.set_ylabel("P(X=k)")
    ax.legend(fontsize=9)

plt.suptitle("Loi binomiale — nombre de fraudes dans un lot", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Pour n=10 000 transactions : E[fraudes] = {10000*p:.1f} ± {np.sqrt(10000*p*(1-p)):.1f}")


---
## 📋 Récapitulatif — Partie 2

| Variable | Loi adaptée | Justification |
|----------|------------|---------------|
| `V1`–`V28` | Normale $\mathcal{N}(0, 1)$ | Issues d'une PCA centrée-réduite |
| `Amount` | Log-normale | Distribution positive, asymétrique à droite |
| `Time` (inter-arrivées) | Exponentielle | Temps entre événements |
| `Class` | Bernoulli | Variable binaire (fraude/normal) |
| Nb fraudes dans un lot | Binomiale | Somme d'épreuves de Bernoulli indépendantes |

**Recommandation :** Appliquer `log(1 + Amount)` avant toute modélisation ML.
